## Load latest headlines

In [1]:
# ========== Bootstrap: ensure required Python packages are present =========
import importlib, subprocess, sys
from typing import Optional

def _ensure(pkg_name: str, import_name: Optional[str] = None):
    try:
        importlib.import_module(import_name or pkg_name)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg_name])
    finally:
        globals()[import_name or pkg_name] = importlib.import_module(import_name or pkg_name)

_ensure('pandas')
print('All dependencies ready.\n')


All dependencies ready.



In [2]:
from pathlib import Path
archive = Path('archive')
archive.mkdir(exist_ok=True)
import pandas as pd
latest = pd.read_csv('../headlines/latest.csv')
latest.head()

,pubdate,source,title,link
0,2026-11-26-20-49-46 +0000,nyt,"U.K. Releases Files on Peter Mandelson, Ambass...",https://www.nytimes.com/2026/11/26/world/europ...
1,2026-11-26-20-45-41 +0000,wapo,"Flight delays, flooding and outages grow as st...",https://www.washingtonpost.com/nation/2026/11/...
2,2026-11-26-20-44-39 +0000,nypost,"Iran imposes electricity rationing, rolling bl...",https://nypost.com/2026/11/26/world-news/rolli...
3,2026-11-26-20-43-17 +0000,nyt,In Praise of My Typewriter,https://www.nytimes.com/2026/11/26/world/ai-ty...
4,2026-11-26-20-42-34 +0000,wapo,Maryland residents urged to conserve water aft...,https://www.washingtonpost.com/nation/2026/11/...


## Calculate word frequencies

In [3]:
import re
from collections import Counter
from datetime import datetime

with open('exclude.txt') as f:
    stop_words = set(w.strip() for w in f if w.strip())
words = re.findall(r'[A-Za-z]+', ' '.join(latest['title']).lower())
filtered = [w for w in words if w not in stop_words and len(w) > 1]
counts = Counter(filtered)
score_df = (
    pd.DataFrame(counts.items(), columns=['word','score'])
    .sort_values('score', ascending=False)
)
score_df[['score','word']].to_csv('scores.csv', index=False)
timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')
score_df[['score','word']].to_csv(archive / f'scores-{timestamp}.csv', index=False)
score_df.head()


/tmp/ipykernel_2715/2220774760.py:15: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')


,word,score
52,thanksgiving,31
61,trump,23
173,new,18
77,he,14
539,china,13


## Rank headlines by score

In [4]:
word_scores = dict(score_df[['word','score']].values)
latest['score'] = latest['title'].apply(
    lambda t: sum(
        word_scores.get(w.lower(), 0)
        for w in re.findall(r'[A-Za-z]+', t)
        if len(w) > 1
    )
)
ranked = latest.sort_values('score', ascending=False)
ranked[['score','pubdate','source','title','link']].to_csv('rank.csv', index=False)
ranked[['score','pubdate','source','title','link']].to_csv(archive / f'rank-{timestamp}.csv', index=False)
ranked.head()


,pubdate,source,title,link,score
7,2026-11-26-20-37-19 +0000,nypost,Trump celebrates Thanksgiving with his holiday...,https://nypost.com/2026/11/26/us-news/trump-ce...,80
111,2026-11-26-14-47-00 +0000,wsj,Macy’s Postpones Thanksgiving Day Parade as Po...,https://www.wsj.com/us-news/macys-postpones-it...,80
122,2026-11-26-13-58-19 +0000,nypost,Trump kicks off Thanksgiving with late night d...,https://nypost.com/2026/11/26/us-news/presiden...,73
16,2026-11-26-20-18-20 +0000,nypost,Rescheduled Macy’s Thanksgiving Day Parade set...,https://nypost.com/2026/11/26/us-news/reschedu...,73
96,2026-11-26-15-39-23 +0000,nypost,Trump tells Putin to ‘pull back all military a...,https://nypost.com/2026/11/26/us-news/trump-te...,70


## Select top headlines

In [5]:
with open('exclude_sources.txt') as f:
    excluded_sources = {
        line.strip()
        for line in f
        if line.strip() and not line.strip().startswith('#')
    }

top_count = 10
top_rows = []
working = word_scores.copy()
remaining = latest.copy()

while len(top_rows) < top_count and not remaining.empty:
    scored_remaining = remaining.assign(
        score=remaining['title'].apply(
            lambda t: sum(
                working.get(w.lower(), 0)
                for w in re.findall(r'[A-Za-z]+', t)
                if len(w) > 1
            )
        )
    )
    ranked_loop = scored_remaining.sort_values('score', ascending=False)

    if ranked_loop.empty:
        break

    top_story = ranked_loop.iloc[0]
    remaining = remaining.drop(top_story.name)

    if top_story['source'] in excluded_sources:
        continue

    words = set(re.findall(r'[A-Za-z]+', top_story['title'].lower()))
    for word in words:
        working.pop(word, None)

    top_rows.append(top_story[['score', 'pubdate', 'source', 'title', 'link']])

top_df = pd.DataFrame(top_rows)
top_df.to_csv('top.csv', index=False)
top_df.to_csv(archive / f'top-{timestamp}.csv', index=False)
top_df


,score,pubdate,source,title,link
7,80,2026-11-26-20-37-19 +0000,nypost,Trump celebrates Thanksgiving with his holiday...,https://nypost.com/2026/11/26/us-news/trump-ce...
202,57,2026-11-26-10-00-00 +0000,wsj,Democratic Sen. Chris Van Hollen wants to be S...,https://www.wsj.com/politics/policy/why-this-d...
5,53,2026-11-26-20-41-23 +0000,nypost,California companies owned by Chinese firms ar...,https://nypost.com/2026/11/26/us-news/chinese-...
302,50,2026-11-25-22-38-54 +0000,nypost,Elon Musk’s plan for SpaceX to launch 1 millio...,https://nypost.com/2026/11/25/business/fcc-app...
187,50,2026-11-26-10-13-24 +0000,nypost,Hong Kong marks anniversary high-rise blaze th...,https://nypost.com/2026/11/26/world-news/hong-...
310,49,2026-11-25-22-21-10 +0000,cbc,"Andy Taylor, guitarist of new wave icons Duran...",https://www.cbc.ca/news/entertainment/duran-du...
111,49,2026-11-26-14-47-00 +0000,wsj,Macy’s Postpones Thanksgiving Day Parade as Po...,https://www.wsj.com/us-news/macys-postpones-it...
13,35,2026-11-26-20-30-14 +0000,nypost,Nvidia halts server shipments to 4 countries j...,https://nypost.com/2026/11/26/business/nvidia-...
96,33,2026-11-26-15-39-23 +0000,nypost,Trump tells Putin to ‘pull back all military a...,https://nypost.com/2026/11/26/us-news/trump-te...
263,32,2026-11-26-02-26-06 +0000,latimes,"Man steals Edison truck, smashes power pole, k...",https://www.latimes.com/california/story/2026-...


## Create JSON version of top headlines

In [6]:
import pandas as pd
pd.read_csv('top.csv').to_json('top.json', orient='records', indent=2)
